# 06. Classification Basics

## 📚 Learning Objectives

By completing this notebook, you will:
- Train and evaluate classifiers (e.g. sklearn)
- Interpret confusion matrix and metrics
- Compare models and feature importance

## 🔗 Prerequisites

- ✅ Basic Python
- ✅ Basic NumPy/Pandas (when applicable)

---

---

## The Story

**BEFORE**: You know linear regression for continuous predictions, but don't know how to predict categories/classes.

**AFTER**: You'll master classification algorithms (Logistic Regression, Decision Trees) to predict categories instead of continuous values!

**Why this matters**: Classification is essential for real-world problems like spam detection, medical diagnosis, and image recognition!

---

# Unit 4 - Example 6: Classification Basics

## 🔗 Solving the Problem from Example 4

**Remember the dead end from Example 4?**
- We learned linear regression for predicting continuous values
- But we discovered we need to predict categories/classes, not continuous values
- Linear regression doesn't work well for classification problems

**This notebook solves that problem!**
- We'll learn **classification algorithms** (Logistic Regression, Decision Trees, etc.)
- We'll learn how to **predict categories** instead of continuous values
- We'll learn **classification metrics** (accuracy, precision, recall, F1-score)

**This solves the classification problem from Example 4!**

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Features, labels
- sklearn

**Outputs:** What you'll see when you run the cells

- Classifier
- Metrics
- Confusion matrix
- Plots

---

In [1]:
# Step 1: Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_curve, roc_auc_score)

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("=" * 70)
print("Example 6: Classification Basics")
print("=" * 70)
print("\n📚 Prerequisites: Example 4 completed, linear regression knowledge")
print("🔗 This is Example 06 in Unit 4 - classification algorithms")
print("🎯 Goal: Master classification with logistic regression and decision trees")

Example 6: Classification Basics

📚 Prerequisites: Example 4 completed, linear regression knowledge
🔗 This is Example 06 in Unit 4 - classification algorithms
🎯 Goal: Master classification with logistic regression and decision trees


1. CREATE CLASSIFICATION DATA


In [2]:
print("\n1. Creating Classification Data")
print("-" * 70)
np.random.seed(42)
n_samples = 500
X1 = np.random.normal(2, 1.5, n_samples)
X2 = np.random.normal(3, 1.5, n_samples)
X = np.column_stack([X1, X2])
y = ((X1 - 2)**2 + (X2 - 3)**2 < 4).astype(int) + np.random.binomial(1, 0.1, n_samples)
y = np.clip(y, 0, 1)
df = pd.DataFrame(X, columns=['feature_1', 'feature_2'])
df['target'] = y
print(f"Data shape: {df.shape}")
print(f"Target distribution:\n{df['target'].value_counts()}")


1. Creating Classification Data
----------------------------------------------------------------------
Data shape: (500, 3)
Target distribution:
target
1    323
0    177
Name: count, dtype: int64


2. LOGISTIC REGRESSION


In [3]:
print("\n\n2. Logistic Regression")
print("-" * 70)
X_data = df[['feature_1', 'feature_2']]
y_data = df['target']
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42, stratify=y_data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
logistic_model = LogisticRegression(random_state=42, max_iter=1000)
logistic_model.fit(X_train_scaled, y_train)
y_test_pred_lr = logistic_model.predict(X_test_scaled)
y_test_proba_lr = logistic_model.predict_proba(X_test_scaled)[:, 1]
accuracy_lr = accuracy_score(y_test, y_test_pred_lr)
print(f"\nLogistic Regression Accuracy: {accuracy_lr:.4f}")



2. Logistic Regression
----------------------------------------------------------------------

Logistic Regression Accuracy: 0.6500


3. DECISION TREE


In [4]:
print("\n\n3. Decision Tree")
print("-" * 70)
tree_model = DecisionTreeClassifier(random_state=42, max_depth=5)
tree_model.fit(X_train, y_train)
y_test_pred_dt = tree_model.predict(X_test)
y_test_proba_dt = tree_model.predict_proba(X_test)[:, 1]
accuracy_dt = accuracy_score(y_test, y_test_pred_dt)
print(f"\nDecision Tree Accuracy: {accuracy_dt:.4f}")

# Feature importance - which feature drives the tree's decisions?
print("\nFeature importance (decision tree):")
for name, imp in zip(X_data.columns, tree_model.feature_importances_):
    print(f"  {name}: {imp:.3f}")
print("(importances sum to 1 - higher = more influence on the splits)")



3. Decision Tree
----------------------------------------------------------------------

Decision Tree Accuracy: 0.8900

Feature importance (decision tree):
  feature_1: 0.477
  feature_2: 0.523
(importances sum to 1 - higher = more influence on the splits)


4. CONFUSION MATRICES


In [5]:
print("\n\n4. Confusion Matrices")
print("-" * 70)
cm_lr = confusion_matrix(y_test, y_test_pred_lr)
cm_dt = confusion_matrix(y_test, y_test_pred_dt)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrices')
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
xticklabels=['Class 0', 'Class 1'], yticklabels=['Class 0', 'Class 1'])
axes[0].set_title('Logistic Regression')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens', ax=axes[1],
xticklabels=['Class 0', 'Class 1'], yticklabels=['Class 0', 'Class 1'])
axes[1].set_title('Decision Tree')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
plt.tight_layout()
plt.savefig('11_confusion_matrices.png', dpi=300, bbox_inches='tight')
print("✓ Confusion matrices saved")
plt.close()



4. Confusion Matrices
----------------------------------------------------------------------
✓ Confusion matrices saved


5. ROC CURVES


In [6]:
print("\n\n5. ROC Curves")
print("-" * 70)
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_test_proba_lr)
auc_lr = roc_auc_score(y_test, y_test_proba_lr)
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_test_proba_dt)
auc_dt = roc_auc_score(y_test, y_test_proba_dt)
plt.figure(figsize=(10, 6))
plt.plot(fpr_lr, tpr_lr, linewidth=2, label=f'Logistic Regression (AUC = {auc_lr:.4f})')
plt.plot(fpr_dt, tpr_dt, linewidth=2, label=f'Decision Tree (AUC = {auc_dt:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('11_roc_curves.png', dpi=300, bbox_inches='tight')
print("✓ ROC curves saved")
plt.close()



5. ROC Curves
----------------------------------------------------------------------


✓ ROC curves saved


6. SUMMARY


In [7]:
print("\n" + "=" * 70)
print("Summary")
print("=" * 70)
print("\nKey Concepts Covered:")
print("1. Logistic Regression for classification")
print("2. Decision Tree classifier")
print("3. Confusion matrix analysis")
print("4. ROC curves and AUC")
print("5. Feature importance from the decision tree")
print("\nNext Steps: Continue to Example 7 for Model Evaluation")



Summary

Key Concepts Covered:
1. Logistic Regression for classification
2. Decision Tree classifier
3. Confusion matrix analysis
4. ROC curves and AUC
5. Feature importance from the decision tree

Next Steps: Continue to Example 7 for Model Evaluation


## 🚫 When Classification Hits a Dead End

**BEFORE**: We've learned to build classification models.

**AFTER**: We discover we need proper evaluation beyond just accuracy!

**Why this matters**: Accuracy alone can be misleading - we need comprehensive evaluation metrics!

---

### The Problem We've Discovered

We've learned:
- ✅ How to build classification models (Logistic Regression, Decision Trees)
- ✅ How to calculate accuracy
- ✅ How to create confusion matrices and ROC curves

**But we have a problem:**
- ❓ **What if accuracy is misleading (imbalanced classes)?**
- ❓ **What if we need to understand model performance in detail?**
- ❓ **What if we need to compare multiple models properly?**

**The Dead End:**
- We can build models and calculate accuracy
- But accuracy alone doesn't tell the full story
- We need comprehensive evaluation metrics and techniques

---

### Demonstrating the Problem

Let's see why accuracy alone can be misleading:

In [8]:
print("\n" + "=" * 70)
print("🚫 DEMONSTRATING THE DEAD END: Accuracy Can Be Misleading")
print("=" * 70)

# Create imbalanced dataset to show the problem
np.random.seed(42)
n_samples = 1000
# Imbalanced: 90% class 0, 10% class 1
y_imbalanced = np.random.choice([0, 1], size=n_samples, p=[0.9, 0.1])
X_imbalanced = np.random.randn(n_samples, 5)

# Dummy classifier that always predicts class 0 (majority class)
y_pred_dummy = np.zeros(n_samples)

accuracy_dummy = accuracy_score(y_imbalanced, y_pred_dummy)
print(f"\n📊 Imbalanced Dataset Example:")
print(f"   - Total samples: {n_samples}")
print(f"   - Class 0 (majority): {(y_imbalanced == 0).sum()} ({(y_imbalanced == 0).sum()/n_samples*100:.1f}%)")
print(f"   - Class 1 (minority): {(y_imbalanced == 1).sum()} ({(y_imbalanced == 1).sum()/n_samples*100:.1f}%)")

print(f"\n⚠️  Dummy Classifier (Always Predicts Class 0):")
print(f"   - Accuracy: {accuracy_dummy:.2%}")
print(f"   - This looks good! But the model is useless!")
print(f"   - It never predicts class 1 (the important class)")

print(f"\n💡 The Problem:")
print(f"   - Accuracy alone can be misleading with imbalanced data")
print(f"   - We need precision, recall, F1-score to understand true performance")
print(f"   - We need to understand trade-offs (precision vs recall)")
print(f"   - We need proper evaluation techniques (cross-validation, learning curves)")

print(f"\n📋 What We Need for Proper Evaluation:")
print(f"   1. Multiple metrics (precision, recall, F1, AUC)")
print(f"   2. Cross-validation (robust performance estimation)")
print(f"   3. Learning curves (understand model behavior)")
print(f"   4. Model comparison (which model is actually better?)")

print(f"\n➡️  Solution Needed:")
print(f"   - We need comprehensive model evaluation techniques")
print(f"   - We need to understand metrics beyond accuracy")
print(f"   - We need proper validation methods")
print(f"   - This leads us to Example 7: Model Evaluation")

print("\n" + "=" * 70)



🚫 DEMONSTRATING THE DEAD END: Accuracy Can Be Misleading

📊 Imbalanced Dataset Example:
   - Total samples: 1000
   - Class 0 (majority): 900 (90.0%)
   - Class 1 (minority): 100 (10.0%)

⚠️  Dummy Classifier (Always Predicts Class 0):
   - Accuracy: 90.00%
   - This looks good! But the model is useless!
   - It never predicts class 1 (the important class)

💡 The Problem:
   - Accuracy alone can be misleading with imbalanced data
   - We need precision, recall, F1-score to understand true performance
   - We need to understand trade-offs (precision vs recall)
   - We need proper evaluation techniques (cross-validation, learning curves)

📋 What We Need for Proper Evaluation:
   1. Multiple metrics (precision, recall, F1, AUC)
   2. Cross-validation (robust performance estimation)
   3. Learning curves (understand model behavior)
   4. Model comparison (which model is actually better?)

➡️  Solution Needed:
   - We need comprehensive model evaluation techniques
   - We need to understan

### What We Need Next

**The Solution**: We need comprehensive model evaluation:
- **Multiple metrics**: Precision, recall, F1-score, AUC (not just accuracy)
- **Cross-validation**: Robust performance estimation
- **Learning curves**: Understand model behavior and overfitting
- **Model comparison**: Proper techniques to compare models

**This dead end leads us to Example 7: Model Evaluation**
- Example 7 will teach us comprehensive evaluation techniques
- We'll learn metrics beyond accuracy
- We'll learn validation methods to properly assess models!
